# 🛠️ Data Cleaning and Integrity Validation
## 📊 Project: Telecom Subscriber Behavior Analysis

---

### 🚀 Introduction
Before conducting statistical tests or building descriptive models, the data must undergo a rigorous **Cleaning** and **Validation** phase. This notebook serves as the foundation for the entire project, ensuring that the insights we derive later are based on accurate, logical, and consistent information.

---

### 🧼 Phase 1: Data Cleaning
**Data Cleaning** is the process of detecting and correcting (or removing) corrupt, inaccurate, or irrelevant records from a dataset. In this project, cleaning involves standardizing the format to make it "analysis-ready" and ensuring that the data reflects the business rules of the telecom industry.



[Image of data cleaning process flowchart]


**Our key cleaning tasks include:**
* **📅 Type Casting:** Converting date strings into `datetime` objects for time-series analysis.
* **❓ Handling Missing Values:** Identifying which null values are informative (like active users) and which are actual data errors.
* **📏 Value Normalization:** Trimming whitespaces from strings and rounding call durations up to the nearest minute (**Ceiling Logic**).

---

### 🛡️ Phase 2: Data Validation
While cleaning focuses on the *format* of the data, **Data Validation** focuses on the *logic* and *integrity* of the information. It answers the fundamental question: *"Does this data represent a logical real-world event?"*

For this telecom dataset, we implement **Temporal Integrity Validation**. This involves cross-referencing activity logs against user profile records to ensure every interaction exists within a valid subscription window.



**We will enforce two primary logical constraints:**
1.  **🚪 The Registration Floor:** Ensuring no activity occurs before a user officially joined the network.
2.  **🚫 The Churn Ceiling:** For users who have canceled their service, ensuring no activity is recorded after their churn date.

---

### ⚠️ Why This Matters
If we skip these steps, our statistical results will be skewed. Data "noise"—such as "zombie" users making calls after a churn date or incorrectly rounded call durations—would lead to inaccurate averages, distorted variances, and flawed business conclusions.

### 📁 1. Environment Setup & Data Connection
In this initial step, we prepare our workspace by importing the necessary libraries and establishing a connection to the data source.

* **Libraries:** We use `pandas` for data manipulation, `numpy` for mathematical operations (like rounding up), and `google.colab` tools for file management.
* **Storage:** By mounting Google Drive, we can access the raw datasets directly and ensure any exported cleaned files are saved securely.

---

In [227]:
import pandas as pd
import numpy as np
from google.colab import files

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 📥 2. Dataset Acquisition
With the environment prepared, we load the raw data into separate DataFrames. Each table represents a distinct dimension of the telecom ecosystem, ranging from user demographics to specific service usage logs.

**Tables Imported:**
* `users`: The master table containing customer IDs, registration dates, and churn information.
* `calls`: Logs of individual voice calls and their durations.
* `messages`: Records of SMS interactions.
* `internet`: Data consumption sessions in megabytes.
* `plans`: Definitions of the different service tiers (Ultimate vs. Surf).

---

In [228]:
# import dataset
users = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/levels of data/level 3/megaline_users.csv')
calls = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/levels of data/level 3/megaline_calls.csv')
messages= pd.read_csv('/content/drive/MyDrive/Colab Notebooks/levels of data/level 3/megaline_messages.csv')
internet = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/levels of data/level 3/megaline_internet.csv')
plans = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/levels of data/level 3/megaline_plans.csv')


### 🔍 3. Initial Exploration & Quality Audit: Users Table
Before modifying any data, we perform a deep audit of the `users` table. This step is vital to establish a baseline for our "Master" table, which contains the ground-truth dates for all other activity logs.

**Key Actions in this Stage:**
* **⚙️ Schema Correction:** Converting date columns from generic objects to `datetime` objects to enable mathematical time-range validation.
* **❓ Missing Value Profiling:** Identifying the extent of nulls—notably in `churn_date`, where nulls represent active customers rather than missing data.
* **🧹 String Sanitization:** Scanning for hidden empty strings or whitespaces in categorical fields like `city` and `plan` that could interfere with future grouping.
* **👤 Demographic Validation:** Auditing the `age` column for biological outliers (ages < 0 or > 100) to ensure the user profiles are realistic.
* **📈 Business Logic Check:** Calculating the current churn rate to understand the proportion of active vs. inactive subscribers in our sample.

---


In [229]:
#basic over view of the data set
print("BASIC OVERVIEW OF USERS DATAFRAME")
print("-" * 50)
print(f"Total rows of users dataset : {len(users)}")
print(f"Total columns of users dataset : {len(users.columns)}")
print(f"Columns of users : {list(users.columns)}")
print(f"Data types : {users.dtypes}")

print("\nDATA DYPES CHANGES ON USERS DATAFRAME")
print("-" * 50)
users['reg_date'] = pd.to_datetime(users['reg_date'])
users['churn_date'] = pd.to_datetime(users['churn_date'])
print(users.dtypes)


# check for Missing values
print("\n\nMISSING VALUES ANALYSIS")
print("-" * 50)
print(users.isnull().sum())

print("\nEMPTY STRINGS AND WHITESPACES")
for col in ['first_name','last_name','city','plan']:
  if col in users.columns and users[col].dtype =='object':
    empty_count = users[col].str.strip().eq('').sum()
    if empty_count >= 0 :
      print(f" {col} : {empty_count} empty/whitespaces strings ")


#duplicate analysis
print("\n\nDUPLICATE ANALYSIS")
print("-" * 50)
print(f"Total duplicate rows : {users.duplicated().sum()}")

#outlier analysis
print("\n\nOutliers ANALYSIS")
print("-" * 50)
print(users['age'].describe())

print("\nAge Analysis:")
invalid_age  = users[(users['age']<0) | (users['age']>100)]['age'].count()
print(f"Total invalid ages : {invalid_age}")


#Business logic validation
print("\n\nBUSSINESS LOGIC VALIDATION")
print("-"*50)
churned_users = users['churn_date'].notna().sum()
active_users = users['churn_date'].isna().sum()
churn_rate = (churned_users /len(users)) * 100

print(f"churn status")
print(f"  Active users: {active_users} ({100-churn_rate:.2f}%)")
print(f"  Churned users: {churned_users} ({churn_rate:.2f}%)")

print("\n\nFINAL RESULTS")
print("-"*50)
users.head()

BASIC OVERVIEW OF USERS DATAFRAME
--------------------------------------------------
Total rows of users dataset : 500
Total columns of users dataset : 8
Columns of users : ['user_id', 'first_name', 'last_name', 'age', 'city', 'reg_date', 'plan', 'churn_date']
Data types : user_id        int64
first_name    object
last_name     object
age            int64
city          object
reg_date      object
plan          object
churn_date    object
dtype: object

DATA DYPES CHANGES ON USERS DATAFRAME
--------------------------------------------------
user_id                int64
first_name            object
last_name             object
age                    int64
city                  object
reg_date      datetime64[ns]
plan                  object
churn_date    datetime64[ns]
dtype: object


MISSING VALUES ANALYSIS
--------------------------------------------------
user_id         0
first_name      0
last_name       0
age             0
city            0
reg_date        0
plan            0
churn

,user_id,first_name,last_name,age,city,reg_date,plan,churn_date
0,1000,Anamaria,Bauer,45,"Atlanta-Sandy Springs-Roswell, GA MSA",2018-12-24,ultimate,NaT
1,1001,Mickey,Wilkerson,28,"Seattle-Tacoma-Bellevue, WA MSA",2018-08-13,surf,NaT
2,1002,Carlee,Hoffman,36,"Las Vegas-Henderson-Paradise, NV MSA",2018-10-21,surf,NaT
3,1003,Reynaldo,Jenkins,52,"Tulsa, OK MSA",2018-01-28,surf,NaT
4,1004,Leonila,Thompson,40,"Seattle-Tacoma-Bellevue, WA MSA",2018-05-23,surf,NaT


### 📞 4. Processing & Policy Enforcement: Calls Table
This stage focuses on the `calls` dataset. Unlike user profiles, this is high-frequency event data, requiring specific attention to the company's billing policies and temporal accuracy.

**Key Actions in this Stage:**
* **🕒 Temporal Formatting:** Converting the `call_date` to a `datetime` object. This is a prerequisite for our upcoming "Two-Gate" temporal validation.
* **⚖️ Billing Policy Implementation (Ceiling Logic):** Applying the company policy where any call duration is rounded up to the nearest whole minute. For example, a call of 0.1 minutes is billed as 1.0 minute. We use `np.ceil()` to ensure this is applied mathematically across all 75,000+ records.
* **🔍 Identifer Audit:** Checking for empty strings or whitespaces in the `id` column to ensure every event is uniquely and cleanly identified.
* **📉 Distribution Analysis:** Auditing the `duration` statistics before and after rounding to understand how the billing policy shifts the total volume of minutes.



---


In [230]:
#Basic overview of the dataframe
print("BASIC OVERVIEW OF CALLS   DATAFRAME")
print("-" * 50)
print(f"Total rows of calls dataset : {len(calls)}")
print(f"Total columns of calls dataset : {len(calls.columns)}")
print(f"Columns of calls : {list(calls.columns)}")
print(f"Data types : \n{calls.dtypes}")

print("\nDATA DYPES CHANGES ON CALLS DATAFRAME")
print("-" * 50)
calls['call_date'] = pd.to_datetime(calls['call_date'])
print(calls.dtypes)

#check for missing values
print("\n\n MISSING VALUES ANALYSIS")
print("-" * 50)
print(calls.isnull().sum())

print("\n CHEKC IF THERE IS EMTPTY STRING IN OJECTS COLUMNS ")
white_spaces_count = (calls['id'].astype(str).str.strip() == '').sum()
print(f"Total white spaces in 'id' column : {white_spaces_count}")

#dplicate analysis
print("\n\nDUPLICATE ANALYSIS")
print("-"*50)
print(f"Total duplicate rows : {calls.duplicated().sum()}")

#outlier analysis
print("\n\nOutliers ANALYSIS")
print("-" * 50)
print("\nDurtion analysis in minutes")
print(calls['duration'].describe())

# round column duration to be all i minutes
print("\nround seconds to be minutes")
calls['duration']= np.ceil(calls['duration'])
print(calls['duration'].describe())

print("\n\nFINAL RESULTS")
print("-"*50)
calls.head()

BASIC OVERVIEW OF CALLS   DATAFRAME
--------------------------------------------------
Total rows of calls dataset : 137735
Total columns of calls dataset : 4
Columns of calls : ['id', 'user_id', 'call_date', 'duration']
Data types : 
id            object
user_id        int64
call_date     object
duration     float64
dtype: object

DATA DYPES CHANGES ON CALLS DATAFRAME
--------------------------------------------------
id                   object
user_id               int64
call_date    datetime64[ns]
duration            float64
dtype: object


 MISSING VALUES ANALYSIS
--------------------------------------------------
id           0
user_id      0
call_date    0
duration     0
dtype: int64

 CHEKC IF THERE IS EMTPTY STRING IN OJECTS COLUMNS 
Total white spaces in 'id' column : 0


DUPLICATE ANALYSIS
--------------------------------------------------
Total duplicate rows : 0


Outliers ANALYSIS
--------------------------------------------------

Durtion analysis in minutes
count    137

,id,user_id,call_date,duration
0,1000_93,1000,2018-12-27,9.0
1,1000_145,1000,2018-12-27,14.0
2,1000_247,1000,2018-12-27,15.0
3,1000_309,1000,2018-12-28,6.0
4,1000_380,1000,2018-12-30,5.0


### 💬 5. Quality Audit: Messages Table
In this section, we analyze the `messages` dataset. While SMS logs are simpler than call records because they lack "duration," they are essential for calculating the total communication volume per user.

**Key Actions in this Stage:**
* **🕒 Time-Series Preparation:** Converting `message_date` into a `datetime` format. This ensures that every text sent can be verified against the user's active subscription period.
* **🆔 Unique Identifier Check:** Scanning the `id` column for any white spaces or empty strings. This ensures that our message counts (GroupBys) will be based on clean, distinct identifiers.
* **🕵️ Structural Integrity:** Performing a duplicate and missing value analysis to confirm that no technical glitches resulted in "phantom" messages or lost data points.
* **📊 Volume Overview:** Establishing the total row count to prepare for the aggregation phase (Messages per user per month).

---

In [231]:
#Basic overview of the dataframe
print("BASIC OVERVIEW OF MESSAGES DATAFRAME")
print("-" * 50)
print(f"Total rows of calls dataset : {len(messages)}")
print(f"Total columns of calls dataset : {len(messages.columns)}")
print(f"Columns of calls : {list(messages.columns)}")
print(f"Data types : \n{messages.dtypes}")

print("\nDATA DYPES CHANGES ON MESSAGES  DATAFRAME")
print("-" * 50)
messages['message_date'] = pd.to_datetime(messages['message_date'])
print(messages.dtypes)

#check for missing values
print("\n\n MISSING VALUES ANALYSIS")
print("-" * 50)
print(messages.isnull().sum())


print("\nCHEKC IF THERE IS EMTPTY STRING IN OJECTS COLUMNS ")
white_spaces_count = (messages['id'].astype(str).str.strip() == '').sum()
print(f"Total white spaces in 'id' column : {white_spaces_count}")


#dplicate analysis
print("\n\nDUPLICATE ANALYSIS")
print("-"*50)
print(f"Total duplicate rows : {messages.duplicated().sum()}")

print("\n\nFINAL RESULTS")
print("-"*50)
messages.head()

BASIC OVERVIEW OF MESSAGES DATAFRAME
--------------------------------------------------
Total rows of calls dataset : 76051
Total columns of calls dataset : 3
Columns of calls : ['id', 'user_id', 'message_date']
Data types : 
id              object
user_id          int64
message_date    object
dtype: object

DATA DYPES CHANGES ON MESSAGES  DATAFRAME
--------------------------------------------------
id                      object
user_id                  int64
message_date    datetime64[ns]
dtype: object


 MISSING VALUES ANALYSIS
--------------------------------------------------
id              0
user_id         0
message_date    0
dtype: int64

CHEKC IF THERE IS EMTPTY STRING IN OJECTS COLUMNS 
Total white spaces in 'id' column : 0


DUPLICATE ANALYSIS
--------------------------------------------------
Total duplicate rows : 0


FINAL RESULTS
--------------------------------------------------


,id,user_id,message_date
0,1000_125,1000,2018-12-27
1,1000_160,1000,2018-12-31
2,1000_223,1000,2018-12-31
3,1000_251,1000,2018-12-27
4,1000_255,1000,2018-12-26


### 🌐 6. Data Integrity Audit: Internet Usage Table
The `internet` table tracks data consumption per session. Since data usage is a primary driver for plan overage charges, ensuring the accuracy of session dates and megabyte counts is a high priority for our statistical model.

**Key Actions in this Stage:**
* **🕒 Chronological Formatting:** Transforming `session_date` into a `datetime` object. This allows us to align data usage sessions with specific billing months.
* **📦 Storage Volume Audit:** Reviewing the `mb_used` column (via `.head()`) to understand the scale of data consumption.
* **🕵️ Connectivity Check:** Identifying any potential duplicates or missing records that could lead to an artificial inflation of a user's data usage profile.
* **🆔 Session Tracking:** Verifying that the `id` column is free of formatting errors (like whitespaces) so that every session is counted uniquely.



---

In [232]:
#Basic overview of the dataframe
print("BASIC OVERVIEW OF INTERNET DATAFRAME")
print("-" * 50)
print(f"Total rows of internet dataset : {len(internet)}")
print(f"Total columns of internet dataset : {len(internet.columns)}")
print(f"Columns of internet : {list(internet.columns)}")
print(f"Data types : \n{internet.dtypes}")


print("\nDATA DYPES CHANGES ON INTERNET  DATAFRAME")
print("-" * 50)
internet['session_date'] = pd.to_datetime(internet['session_date'])
print(internet.dtypes)

#check for missing values
print("\n\n INTERNET VALUES ANALYSIS")
print("-" * 50)
print(internet.isnull().sum())


print("\nCHEKC IF THERE IS EMTPTY STRING IN OJECTS COLUMNS ")
white_spaces_count = (internet['id'].astype(str).str.strip() == '').sum()
print(f"Total white spaces in 'id' column : {white_spaces_count}")


#dplicate analysis
print("\n\nDUPLICATE ANALYSIS")
print("-"*50)
print(f"Total duplicate rows : {internet.duplicated().sum()}")

print("\n\nFINAL RESULTS")
print("-"*50)
internet.head()

BASIC OVERVIEW OF INTERNET DATAFRAME
--------------------------------------------------
Total rows of internet dataset : 104825
Total columns of internet dataset : 4
Columns of internet : ['id', 'user_id', 'session_date', 'mb_used']
Data types : 
id               object
user_id           int64
session_date     object
mb_used         float64
dtype: object

DATA DYPES CHANGES ON INTERNET  DATAFRAME
--------------------------------------------------
id                      object
user_id                  int64
session_date    datetime64[ns]
mb_used                float64
dtype: object


 INTERNET VALUES ANALYSIS
--------------------------------------------------
id              0
user_id         0
session_date    0
mb_used         0
dtype: int64

CHEKC IF THERE IS EMTPTY STRING IN OJECTS COLUMNS 
Total white spaces in 'id' column : 0


DUPLICATE ANALYSIS
--------------------------------------------------
Total duplicate rows : 0


FINAL RESULTS
--------------------------------------------

,id,user_id,session_date,mb_used
0,1000_13,1000,2018-12-29,89.86
1,1000_204,1000,2018-12-31,0.00
2,1000_379,1000,2018-12-28,660.40
3,1000_413,1000,2018-12-26,270.99
4,1000_442,1000,2018-12-27,880.22


### 📋 7. Static Reference: Plans Table
The `plans` table serves as a fixed reference for our analysis. It defines the constraints and pricing for the "Surf" and "Ultimate" tiers.

**Observation:**
* **No Cleaning Required:** The dataset is small, structured, and contains static categorical information.
* **Data Utility:** We will use the `mb_per_month_included`, `minutes_included`, and `messages_included` columns from this table later to calculate overage costs and user behavior relative to their limits.

---

In [233]:
plans.info()
plans.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   messages_included      2 non-null      int64  
 1   mb_per_month_included  2 non-null      int64  
 2   minutes_included       2 non-null      int64  
 3   usd_monthly_pay        2 non-null      int64  
 4   usd_per_gb             2 non-null      int64  
 5   usd_per_message        2 non-null      float64
 6   usd_per_minute         2 non-null      float64
 7   plan_name              2 non-null      object 
dtypes: float64(2), int64(5), object(1)
memory usage: 260.0+ bytes


,messages_included,mb_per_month_included,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,plan_name
0,50,15360,500,20,10,0.03,0.03,surf
1,1000,30720,3000,70,7,0.01,0.01,ultimate


### 🛡️ 8. Logical Validation: User Account Timelines
Before cross-referencing activity across tables, we must validate the internal logic of the `users` table itself. The most fundamental rule of a customer lifecycle is that an account cannot be closed before it has been opened.

**The "Impossible Timeline" Check:**
* **Logic:** We identify any records where the `churn_date` occurs chronologically *before* the `reg_date`.
* **Action:** Any user falling into this category represents a critical system error or data corruption. These records are flagged and removed from the dataset to prevent logical contradictions in our final analysis.



---

In [234]:
invalid_dates = users[users['churn_date'].notna() &(users['churn_date']<users['reg_date'])]
print(f" found {len(invalid_dates)} users with invalid dates")

if len(invalid_dates) >0 :
  print("\n printing invalid dates")
  print(invalid_dates[['user_id','reg_date','churn_date']])
  #remove invalid records
  users = users = users[~((users['churn_date'].notna()) & (users['churn_date'] < users['reg_date']))]

 found 0 users with invalid dates


### 🛡️ 9. Cross-Table Temporal Integrity Validation
This is the most critical stage of our data validation. We establish a "Single Source of Truth" by ensuring that every interaction—whether a call, a text, or a data session—happened while the user was actually an active subscriber.

**The "Two-Gate" Validation Logic:**
To maintain absolute data integrity, we define a custom validation function that enforces two strict temporal boundaries for every activity record:

1.  **🚪 The Registration Gate:** Activity date must be greater than or equal to the `reg_date`. (No "Pre-subscriber" usage).
2.  **🚫 The Churn Gate:** If a user has churned, the activity date must be less than or equal to the `churn_date`. (No "Zombie" usage).



**Implementation Process:**
* **Dynamic Merging:** The function temporarily joins each activity table with the master `users` table to fetch the valid date ranges for each specific `user_id`.
* **Conflict Identification:** We filter for records that violate either the "Floor" (Registration) or the "Ceiling" (Churn) logic.
* **Automated Cleaning:** The function generates a detailed report of the invalid entries and returns a "Clean" version of the DataFrame, stripping out the illogical records.

This process ensures that our subsequent statistical analysis is based purely on verified, real-world subscriber behavior.

---

In [235]:
def validate_dates_against_user_date_range(activaty_df,users_df,date_column,activaty_name):
  #as columns alreagy converted to dates , there is no need to do it here
  #let's merge tables
  merged = activaty_df.merge(
      users_df[['user_id','reg_date','churn_date']],
      on ='user_id',
      how='left'
  )
  #check invalid dates
  invalid_before_reg_date = merged[merged[date_column] < merged['reg_date']]
  invalid_after_churn_date= merged[merged['churn_date'].notna() & (merged[date_column] > merged['churn_date'])]

  total_invalid = len(invalid_before_reg_date) + len(invalid_after_churn_date)
  print(f"\n{'='*50}")
  print(f"VALIDATION: {activaty_name.upper()}")
  print(f"{'='*50}")
  print(f"Total {activaty_name}: {len(activaty_df)}")
  print(f"Invalid {activaty_name} before registration: {len(invalid_before_reg_date)}")
  print(f"Invalid {activaty_name} after churn: {len(invalid_after_churn_date)}")
  print(f"Total invalid: {total_invalid}")
  if len(invalid_before_reg_date) > 0:
        print(f"\nSample of {activaty_name} before registration:")
        print(invalid_before_reg_date[[date_column, 'reg_date', 'user_id']].head())

  if len(invalid_after_churn_date) > 0:
    print(f"\nSample of {activaty_name} after churn:")
    print(invalid_after_churn_date[[date_column, 'churn_date', 'user_id']].head())
  # Return cleaned dataframe (remove invalid records)
    valid_df = activaty_df[
        ~activaty_df.index.isin(invalid_before_reg_date.index) &
        ~activaty_df.index.isin(invalid_after_churn_date.index)
    ]

    print(f"\nAfter cleaning: {len(valid_df)} valid {activaty_df}")

    return valid_df

#Apply this function on all these functions

# 1. Validate calls
print("\n" + "="*50)
print("VALIDATING ALL ACTIVITY TABLES AGAINST USER PERIODS")
print("="*50)

calls_clean = validate_dates_against_user_date_range(
    calls,
    users,
    date_column='call_date',
    activaty_name='calls'
)

# 2. Validate messages
messages_clean = validate_dates_against_user_date_range(
    messages,
    users,
    date_column='message_date',
    activaty_name='messages'
)

# 3. Validate internet sessions
internet_clean = validate_dates_against_user_date_range(
    internet,
    users,
    date_column='session_date',
    activaty_name='internet sessions'
)

print("\n" + "="*50)
print("VALIDATION COMPLETE")
print("="*50)


VALIDATING ALL ACTIVITY TABLES AGAINST USER PERIODS

VALIDATION: CALLS
Total calls: 137735
Invalid calls before registration: 0
Invalid calls after churn: 2961
Total invalid: 2961

Sample of calls after churn:
      call_date churn_date  user_id
973  2018-12-21 2018-12-18     1006
976  2018-12-26 2018-12-18     1006
3561 2018-12-19 2018-11-16     1012
3562 2018-12-23 2018-11-16     1012
3563 2018-12-07 2018-11-16     1012

After cleaning: 134774 valid               id  user_id  call_date  duration
0        1000_93     1000 2018-12-27       9.0
1       1000_145     1000 2018-12-27      14.0
2       1000_247     1000 2018-12-27      15.0
3       1000_309     1000 2018-12-28       6.0
4       1000_380     1000 2018-12-30       5.0
...          ...      ...        ...       ...
137730  1499_199     1499 2018-11-21       9.0
137731  1499_200     1499 2018-10-20      11.0
137732  1499_201     1499 2018-09-21       9.0
137733  1499_202     1499 2018-10-10       1.0
137734  1499_203     1499 

### 📊 10. Data Integrity Summary
The final stage of our cleaning process is a comparative audit to measure the impact of our validation rules. By quantifying the delta between the raw and cleaned datasets, we can assess the overall health of the original data.

**What this summary reveals:**
* **Data Loss Metric:** We track exactly how many records were removed due to temporal inconsistencies (Pre-registration or Post-churn activity).
* **Dataset Reliability:** A low number of removed records confirms high data quality, while a high number would suggest a need to investigate the system's logging mechanisms.
* **Final Analysis Volume:** This provides the definitive row counts that will be used in our subsequent statistical modeling and hypothesis testing.



---

In [236]:
validation_results = pd.DataFrame({
    'Data Name': ['Calls', 'Messages', 'Internet Sessions'],
    'Count Before Validation': [len(calls), len(messages), len(internet)],
    'Count After Validation': [len(calls_clean), len(messages_clean), len(internet_clean)]
})


validation_results['Records Removed'] = (
    validation_results['Count Before Validation'] -
    validation_results['Count After Validation']
)

print("\n" + "="*50)
print("VALIDATION SUMMARY")
print("="*50)
print(validation_results.to_string(index=False))


VALIDATION SUMMARY
        Data Name  Count Before Validation  Count After Validation  Records Removed
            Calls                   137735                  134774             2961
         Messages                    76051                   74460             1591
Internet Sessions                   104825                  102362             2463


### 💾 11. Final Export & Data Persistence
The data cleaning and validation phase is now complete. To ensure our next notebook (Statistical Analysis) starts with a clean slate, we export the verified DataFrames into new CSV files.

**Export Strategy:**
* **📦 Clean Versioning:** We save the activity tables (Calls, Messages, Internet) and the Users table with a `_clean` suffix to distinguish them from the raw source files.
* **💾 Local & Cloud Backup:** By using `files.download()`, we secure a local copy of the processed data, while the mounted drive ensures the work is persisted in our project environment.
* **✅ Ready for Analysis:** These files now contain 100% logically consistent dates and rounded billing values, making them ready for statistical aggregation and hypothesis testing.

---
### 🏁 Conclusion
The foundation is now laid. We have moved from raw, unverified logs to a refined, high-integrity dataset. We can proceed to the next phase of the project with full confidence in the accuracy of our numbers.

In [237]:
# save cleaned dataset and rename the others
calls_clean.to_csv('calls_clean.csv', index=False)
messages_clean.to_csv('messages_clean.csv', index=False)
internet_clean.to_csv('internet_clean.csv', index=False)
users.to_csv('users_clean.csv', index=False)
plans.to_csv('plans_clean.csv')


files.download('calls_clean.csv')
files.download('messages_clean.csv')
files.download('internet_clean.csv')
files.download('users_clean.csv')
files.download('plans_clean.csv')

print("✓ All files saved and downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ All files saved and downloaded!
